## api.open-elevation.com

In [ ]:
import requests

locations = [
    {"latitude": 13.0827, "longitude": 77.5877},
    {"latitude": 12.9716, "longitude": 77.5946},
]

response = requests.post(
    "https://api.open-elevation.com/api/v1/lookup",
    json={"locations": locations},
)
results = response.json()["results"]

for pt in results:
    print(f"Lat: {pt['latitude']}, Lon: {pt['longitude']} -> {pt['elevation']}m")

Lat: 13.0827, Lon: 77.5877 -> 916.0m
Lat: 12.9716, Lon: 77.5946 -> 911.0m


In [ ]:
import numpy as np
import pandas as pd
import requests


def find_highest_elevation_open_elevation(
    min_lat, max_lat, min_lon, max_lon, grid_resolution=20, batch_size=100
):
    """Creates a grid of coordinates and queries Open-Elevation via JSON POST in batches."""
    # 1. Generate 2D grid of coordinates
    lats = np.linspace(min_lat, max_lat, grid_resolution)
    lons = np.linspace(min_lon, max_lon, grid_resolution)
    grid_lats, grid_lons = np.meshgrid(lats, lons)

    flat_lats = grid_lats.flatten()
    flat_lons = grid_lons.flatten()

    all_elevations = []

    url = "https://api.open-elevation.com/api/v1/lookup"

    # 2. Query in batches using POST
    total_points = len(flat_lats)
    for i in range(0, total_points, batch_size):
        batch_lats = flat_lats[i : i + batch_size]
        batch_lons = flat_lons[i : i + batch_size]

        # Prepare JSON payload
        locations_payload = {
            "locations": [
                {"latitude": round(float(lat), 6), "longitude": round(float(lon), 6)}
                for lat, lon in zip(batch_lats, batch_lons)
            ]
        }

        response = requests.post(url, json=locations_payload, timeout=20)
        if response.status_code != 200:
            raise Exception(
                f"API Request failed with status code: {response.status_code}"
            )

        data = response.json()
        results = data.get("results", [])

        batch_elevations = [item["elevation"] for item in results]
        all_elevations.extend(batch_elevations)

    # 3. Construct DataFrame and sort
    df = pd.DataFrame(
        {
            "latitude": flat_lats,
            "longitude": flat_lons,
            "elevation": all_elevations,
        }
    )

    df_sorted = df.sort_values(by="elevation", ascending=False).reset_index(
        drop=True
    )
    highest_point = df_sorted.iloc[0]

    return highest_point, df_sorted


# =====================================================================
# BANGALORE BOUNDING BOX
# =====================================================================
BANGALORE_BOUNDS = {
    "name": "Greater Bangalore Metropolitan Region",
    "min_lat": 12.80,
    "max_lat": 13.15,
    "min_lon": 77.45,
    "max_lon": 77.75,
}

print(f"Scanning grid across {BANGALORE_BOUNDS['name']}...")

# Run 20x20 grid search (400 points)
peak, sorted_grid = find_highest_elevation_open_elevation(
    min_lat=BANGALORE_BOUNDS["min_lat"],
    max_lat=BANGALORE_BOUNDS["max_lat"],
    min_lon=BANGALORE_BOUNDS["min_lon"],
    max_lon=BANGALORE_BOUNDS["max_lon"],
    grid_resolution=20,
    batch_size=100,
)

print("\n=== HIGHEST ELEVATION POINT IN BANGALORE ===")
print(f"Latitude:  {peak['latitude']:.6f}°")
print(f"Longitude: {peak['longitude']:.6f}°")
print(
    f"Elevation: {peak['elevation']} meters (~{int(peak['elevation'] * 3.28084)} feet)"
)

print("\n=== TOP 5 ELEVATED SPOTS IN THE GRID ===")
print(sorted_grid.head(5).to_string(index=False))

Scanning grid across Greater Bangalore Metropolitan Region...

=== HIGHEST ELEVATION POINT IN BANGALORE ===
Latitude:  12.818421°
Longitude: 77.576316°
Elevation: 951.0 meters (~3120 feet)

=== TOP 5 ELEVATED SPOTS IN THE GRID ===
 latitude  longitude  elevation
12.818421  77.576316      951.0
12.836842  77.592105      946.0
13.076316  77.576316      946.0
12.984211  77.592105      943.0
12.836842  77.623684      940.0


## api.opentopodata.org

In [12]:
import numpy as np
import pandas as pd
import requests


def find_highest_elevation_opentopodata(
    min_lat, max_lat, min_lon, max_lon, grid_resolution=15, batch_size=100
):
    lats = np.linspace(min_lat, max_lat, grid_resolution)
    lons = np.linspace(min_lon, max_lon, grid_resolution)
    grid_lats, grid_lons = np.meshgrid(lats, lons)

    flat_lats = grid_lats.flatten()
    flat_lons = grid_lons.flatten()

    all_elevations = []
    url = "https://api.opentopodata.org/v1/srtm30m"

    # Query in batches
    for i in range(0, len(flat_lats), batch_size):
        batch_lats = flat_lats[i : i + batch_size]
        batch_lons = flat_lons[i : i + batch_size]

        loc_str = "|".join(
            f"{lat:.6f},{lon:.6f}" for lat, lon in zip(batch_lats, batch_lons)
        )

        response = requests.get(f"{url}?locations={loc_str}", timeout=20)
        if response.status_code != 200:
            raise Exception(
                f"API Request failed with status code: {response.status_code}"
            )

        results = response.json().get("results", [])
        all_elevations.extend([item["elevation"] for item in results])

    df = pd.DataFrame(
        {
            "latitude": flat_lats,
            "longitude": flat_lons,
            "elevation": all_elevations,
        }
    )
    df_sorted = df.sort_values(by="elevation", ascending=False).reset_index(
        drop=True
    )

    return df_sorted.iloc[0], df_sorted


# Test OpenTopoData
peak, sorted_grid = find_highest_elevation_opentopodata(
    12.80, 13.15, 77.45, 77.75, grid_resolution=15
)
print(f"Peak Elevation: {peak['elevation']} meters at ({peak['latitude']}, {peak['longitude']})")

Peak Elevation: 948.0 meters at (13.075000000000001, 77.57857142857144)


## api.open-meteo.com

In [14]:
import numpy as np
import pandas as pd
import requests


def find_highest_elevation_open_meteo(
    min_lat, max_lat, min_lon, max_lon, grid_resolution=20, batch_size=100
):
    """Creates a grid of coordinates and queries Open-Meteo via GET in batches (max 100 per call)."""
    # 1. Generate 2D grid of coordinates
    lats = np.linspace(min_lat, max_lat, grid_resolution)
    lons = np.linspace(min_lon, max_lon, grid_resolution)
    grid_lats, grid_lons = np.meshgrid(lats, lons)

    flat_lats = grid_lats.flatten()
    flat_lons = grid_lons.flatten()

    all_elevations = []
    url = "https://api.open-meteo.com/v1/elevation"

    # 2. Query in batches of <= 100
    total_points = len(flat_lats)
    for i in range(0, total_points, batch_size):
        batch_lats = flat_lats[i : i + batch_size]
        batch_lons = flat_lons[i : i + batch_size]

        # Comma-separated coordinate strings
        lat_str = ",".join(f"{lat:.6f}" for lat in batch_lats)
        lon_str = ",".join(f"{lon:.6f}" for lon in batch_lons)

        params = {"latitude": lat_str, "longitude": lon_str}

        response = requests.get(url, params=params, timeout=20)

        if response.status_code != 200:
            raise Exception(
                f"API Request failed with status code {response.status_code}: {response.text}"
            )

        data = response.json()
        all_elevations.extend(data.get("elevation", []))

    # 3. Construct DataFrame and sort
    df = pd.DataFrame(
        {
            "latitude": flat_lats,
            "longitude": flat_lons,
            "elevation": all_elevations,
        }
    )

    df_sorted = df.sort_values(by="elevation", ascending=False).reset_index(
        drop=True
    )
    highest_point = df_sorted.iloc[0]

    return highest_point, df_sorted


# Bangalore Bounding Box
BANGALORE_BOUNDS = {
    "min_lat": 12.80,
    "max_lat": 13.15,
    "min_lon": 77.45,
    "max_lon": 77.75,
}

print("Scanning grid across Bangalore using Open-Meteo API...")
peak, sorted_grid = find_highest_elevation_open_meteo(
    min_lat=BANGALORE_BOUNDS["min_lat"],
    max_lat=BANGALORE_BOUNDS["max_lat"],
    min_lon=BANGALORE_BOUNDS["min_lon"],
    max_lon=BANGALORE_BOUNDS["max_lon"],
    grid_resolution=20,
    batch_size=100,  # Strict cap per Open-Meteo docs
)

print("\n=== HIGHEST ELEVATION POINT IN BANGALORE ===")
print(f"Latitude:  {peak['latitude']:.6f}°")
print(f"Longitude: {peak['longitude']:.6f}°")
print(
    f"Elevation: {peak['elevation']} meters (~{int(peak['elevation'] * 3.28084)} feet)"
)

print("\n=== TOP 5 ELEVATED SPOTS IN THE GRID ===")
print(sorted_grid.head(5).to_string(index=False))

Scanning grid across Bangalore using Open-Meteo API...

=== HIGHEST ELEVATION POINT IN BANGALORE ===
Latitude:  12.836842°
Longitude: 77.623684°
Elevation: 946.0 meters (~3103 feet)

=== TOP 5 ELEVATED SPOTS IN THE GRID ===
 latitude  longitude  elevation
12.836842  77.623684      946.0
13.076316  77.576316      946.0
12.836842  77.592105      945.0
12.836842  77.655263      940.0
12.818421  77.576316      938.0
